In [1]:
import pandas as pd

csv_players = "../data/raw/atp_players.csv"

In [2]:
#Busca o id do big3

df_players = pd.read_csv(csv_players)

players = [
    ("Roger", "Federer"),
    ("Rafael", "Nadal"),
    ("Novak", "Djokovic")
]

df_ids = df_players[
    df_players[["name_first","name_last"]].apply(tuple, axis=1).isin(players)
]

ids = df_ids["player_id"].tolist()

/tmp/ipykernel_29554/1100796783.py:3: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  df_players = pd.read_csv(csv_players)


In [4]:
# salvar as vitorias / derrotas do big3

stats = {
    pid: {
        "clay_w": 0, "clay_l": 0,
        "grass_w": 0, "grass_l": 0,
        "hard_w": 0, "hard_l": 0
    }
    for pid in ids
    # achei essa forma interessante de fazer, me lembrou algo de C
}

csv_matches = "../data/raw/atp_matches_"

for i in range(27):
    ano = 1998 + i
    csv_atual = f"{csv_matches}{ano}.csv"
    
    df_matches = pd.read_csv(csv_atual)

    for _, row in df_matches.iterrows():
        w = row["winner_id"]
        l = row["loser_id"]
        s = row["surface"]

        # Vitória
        if w in stats:
            if s == "Clay":
                stats[w]["clay_w"] += 1
            elif s == "Grass":
                stats[w]["grass_w"] += 1
            elif s == "Hard":
                stats[w]["hard_w"] += 1

        # Derrota
        if l in stats:
            if s == "Clay":
                stats[l]["clay_l"] += 1
            elif s == "Grass":
                stats[l]["grass_l"] += 1
            elif s == "Hard":
                stats[l]["hard_l"] += 1

df_final = pd.DataFrame.from_dict(stats, orient="index")
df_final.reset_index(inplace=True)
df_final.rename(columns={"index": "player_id"}, inplace=True)

df_final = df_final.merge(
    df_ids[["player_id", "name_last"]],
    on="player_id",
    how="left"
)

df_final.to_csv("../data/clean/big3_matches.csv", index=False)

In [ ]:
import pandas as pd

csv_big3_matches = "../data/clean/big3_matches.csv"
df = pd.read_csv(csv_big3_matches)

stats = {}

for _, row in df.iterrows():
    pid = row["player_id"]

    w_clay = row["clay_w"]
    d_clay = row["clay_l"]

    w_grass = row["grass_w"]
    d_grass = row["grass_l"]

    w_hard = row["hard_w"]
    d_hard = row["hard_l"]

    stats[pid] = {
        "clay": w_clay / (w_clay + d_clay) if (w_clay + d_clay) > 0 else 0,
        "grass": w_grass / (w_grass + d_grass) if (w_grass + d_grass) > 0 else 0,
        "hard": w_hard / (w_hard + d_hard) if (w_hard + d_hard) > 0 else 0
    }

df_aux = pd.DataFrame.from_dict(stats, orient="index")
df_aux.reset_index(inplace=True)
df_aux.rename(columns={"index": "player_id"}, inplace=True)

df_final = df.merge(df_aux, on="player_id", how="left")

df_final.to_csv("../data/clean/big3_matches.csv", index=False)